In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Dataset/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')

In [3]:
import pandas as pd
import numpy as np

# 1. 기본 정보
print("shape:", df.shape)
print("\n[컬럼별 타입]")
print(df.dtypes)

# 2. 결측값 확인
print("\n[결측값 개수]")
print(df.isnull().sum())

print("\n[결측값 비율]")
print((df.isnull().mean() * 100).round(2))

# 3. 중복 행 확인
print("\n[전체 중복 행 개수]")
print(df.duplicated().sum())

# 4. ID 중복 확인
id_cols = ['transaction_id', 'user_id']

for col in id_cols:
    if col in df.columns:
        print(f"\n[{col} 중복 개수]")
        print(df[col].duplicated().sum())

# 5. 이상한 문자열 값 확인
cat_cols = df.select_dtypes(include='object').columns

print("\n[범주형 컬럼별 고유값]")
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

# 6. 공백 문자열 확인
print("\n[공백 문자열 개수]")
for col in cat_cols:
    blank_count = (df[col].astype(str).str.strip() == '').sum()
    if blank_count > 0:
        print(col, blank_count)

# 7. 숫자형 컬럼 통계 확인
num_cols = df.select_dtypes(include=np.number).columns

print("\n[숫자형 통계]")
print(df[num_cols].describe())

# 8. 음수값 확인
print("\n[음수값 개수]")
for col in num_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(col, neg_count)

# 9. 말이 안 되는 값 범위 확인
print("\n[비정상 범위 체크]")

checks = {
    'age': (0, 100),
    'daily_screen_time_hours': (0, 24),
    'social_media_hours': (0, 24),
    'gaming_hours': (0, 24),
    'work_study_hours': (0, 24),
    'sleep_hours': (0, 24),
    'notifications_per_day': (0, 1000),
    'app_opens_per_day': (0, 1000),
    'weekend_screen_time': (0, 24),
    'addicted_label': (0, 1)
}

for col, (low, high) in checks.items():
    if col in df.columns:
        bad = df[(df[col] < low) | (df[col] > high)]
        print(f"{col}: 비정상 {len(bad)}개")

# 10. 논리적으로 이상한 값 확인
print("\n[논리 오류 체크]")

if {'social_media_hours', 'gaming_hours', 'work_study_hours', 'daily_screen_time_hours'}.issubset(df.columns):
    df['sum_usage_parts'] = (
        df['social_media_hours'] +
        df['gaming_hours'] 
    )

    logic_error = df[df['sum_usage_parts'] > df['daily_screen_time_hours']]
    print("세부 사용시간 합 > daily_screen_time_hours:", len(logic_error))

# 11. addiction_level과 addicted_label 불일치 확인
print("\n[타겟 관련 이상값 체크]")

if {'addiction_level', 'addicted_label'}.issubset(df.columns):
    print("\naddiction_level별 addicted_label 분포:")
    print(pd.crosstab(df['addiction_level'], df['addicted_label'], dropna=False))

# 12. 이상치 IQR 방식
print("\n[IQR 이상치 개수]")
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)}개")

shape: (7500, 16)

[컬럼별 타입]
transaction_id              object
user_id                     object
age                          int64
gender                      object
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day        int64
app_opens_per_day            int64
weekend_screen_time        float64
stress_level                object
academic_work_impact        object
addiction_level             object
addicted_label               int64
dtype: object

[결측값 개수]
transaction_id               0
user_id                      0
age                          0
gender                       0
daily_screen_time_hours      0
social_media_hours           0
gaming_hours                 0
work_study_hours             0
sleep_hours                  0
notifications_per_day        0
app_opens_per_day            0
weekend_screen_time          0
stress_level    